# Data Preparation

## What this notebook does
1. Loads the VAE latent vectors (β = 3×10⁻⁵) produced by the training pipeline  
2. Loads and preprocesses PPMI clinical assessment tables  
3. Computes UPDRS subscores and total scores  
4. Merges clinical data with latent vectors on `PATNO + EVENT_ID`  
5. Applies a **patient-stratified** train/val split (fixes data leakage in baseline)  
6. Fits the SBR PCA **once** on the training set and saves it for reuse  
7. Saves the final train and validation files  

## Improvements over baseline (Mahmoud's 5.0)
- Patient-stratified split: all visits of one patient stay in one split  
- Clinical scores added: UPDRS I–IV, MoCA, disease duration, REM sleep  
- SBR PCA fitted once and saved — not recomputed in every notebook  
- Explicit coverage report: how many rows have each clinical variable  


## 1. Imports and Configuration

In [34]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
import glob
warnings.filterwarnings('ignore')

# Paths
# Latent vectors (VAE output — beta = 3e-5)
LATENT_FILE = '../../data/baseline/final_train_combined_vae_data.csv'

# Raw PPMI clinical tables
CLINICAL_DIR = '../../data/raw/ppmi_clinical'

UPDRS_I_FILE   = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_II_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_III_FILE = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_III-Archived_25Jun2026.csv')
UPDRS_IV_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_IV-Archived_25Jun2026.csv')
MOCA_FILE      = os.path.join(CLINICAL_DIR, 'Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv')
PD_FEAT_FILE   = os.path.join(CLINICAL_DIR, 'PD_Features-Archived_25Jun2026.csv')
REM_FILE       = os.path.join(CLINICAL_DIR, 'REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv')

# Output paths
TRAIN_OUT  = '../../data/processed/clinical_merged/train.csv'
VAL_OUT    = '../../data/processed/clinical_merged/val.csv'
SCALER_OUT = '../../results/models/scaler_sbr.pkl'
PCA_OUT    = '../../results/models/pca_sbr.pkl'

# Parameters
TRAIN_RATIO        = 0.8
RANDOM_STATE       = 42
VARIANCE_THRESHOLD = 0.1   # latent dims with std below this treated as collapsed
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]
N_SBR_PCS   = 3
PATIENT_COL = 'PATNO'
LABEL_COL   = 'label'
JOIN_KEY    = ['PATNO', 'EVENT_ID']

# Create output directories
os.makedirs('../../data/processed/clinical_merged', exist_ok=True)
os.makedirs('../../results/models', exist_ok=True)

print("Configuration loaded.")
print(f"  Train ratio:        {TRAIN_RATIO}")
print(f"  Random state:       {RANDOM_STATE}")
print(f"  SBR columns:        {len(SBR_COLS)}")
print(f"  SBR PCs to extract: {N_SBR_PCS}")


# Check if the clinical data files exist

tables = {
    "MDS_UPDRS_Part_I":              UPDRS_I_FILE,
    "MDS_UPDRS_Part_II":             UPDRS_II_FILE,
    "MDS_UPDRS_Part_III":            UPDRS_III_FILE,
    "MDS_UPDRS_Part_IV":             UPDRS_IV_FILE,
    "Montreal_Cognitive_Assessment": MOCA_FILE,
    "PD_Features":                   PD_FEAT_FILE,
    "REM_Sleep":                     REM_FILE,
}

print("-" * 65)     
print("Checking for clinical data files...")
for name, pattern in tables.items():
    matches = glob.glob(pattern)
    if matches:
        df = pd.read_csv(matches[0], nrows=2)
        print(f" File:    {matches[0]}")
    else:
        print(f"✗ {name} — not found")
        print()

Configuration loaded.
  Train ratio:        0.8
  Random state:       42
  SBR columns:        6
  SBR PCs to extract: 3
-----------------------------------------------------------------
Checking for clinical data files...
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_III-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_IV-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/PD_Features-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv


## 2. Load Latent Vector

In [20]:
# Load the latent vectors from the baseline(Mahmoud's) VAE model
df_latents = pd.read_csv(LATENT_FILE)

print(f"Shape:           {df_latents.shape}")
print(f"Unique patients: {df_latents[PATIENT_COL].nunique()}")
print(f"Total rows:      {len(df_latents)}")
print(f"\nLabel counts:")
print(df_latents[LABEL_COL].value_counts().to_string())
print(f"\nEVENT_ID distribution:")
print(df_latents['EVENT_ID'].value_counts().to_string())
print(f"\nRows per patient (mean): {len(df_latents) / df_latents[PATIENT_COL].nunique():.2f}")

Shape:           (2373, 303)
Unique patients: 1437
Total rows:      2373

Label counts:
label
PD         2030
Control     233
SWEDD       110

EVENT_ID distribution:
EVENT_ID
SC     1228
V06     392
V04     385
V10     256
U01      41
ST       32
V02      23
V05      10
U02       6

Rows per patient (mean): 1.65
